[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week2_drift_detection_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 2 -- The Model Is Drifting (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** Population Stability Index, KS test, KL-divergence, Evidently AI drift reports

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Compute PSI for all CreditDefault-NG features across the Q1-to-Q4 time split
- Run the KS test and apply Bonferroni correction for multiple testing
- Generate an Evidently AI drift report comparing training and current data
- Classify each feature as stable (PSI < 0.1), moderate (0.1-0.25), or severe (PSI > 0.25)
- Write the drift summary for Farida: no statistical jargon, impact in business terms



---

## MONDAY -- "Population Stability Index"


PSI measures how much a feature distribution has shifted between two populations. It is the standard drift metric in financial services. PSI < 0.1: stable. PSI 0.1-0.25: moderate, investigate. PSI > 0.25: severe, model reliability at risk. PSI is computed on binned distributions -- you bin the feature range into N intervals, compute the fraction of each population in each bin, then sum the weighted KL-style divergence.

Pause and Predict -- before computing: which CreditDefault-NG feature do you expect to have the highest PSI between Q1 and Q4? The credit limit (LIMIT_BAL), the most recent payment status (PAY_0), or the most recent bill amount (BILL_AMT1)? Write your prediction before running.


### Exercise 2.1 -- PSI all features

Compute PSI for every feature. Build a sorted dataframe: feature, PSI, severity. Plot the Q1 vs Q4 histogram for the three highest-PSI features.


In [ ]:
# Exercise 2.1: PSI all features
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


### Exercise 2.2 -- PSI context

For the three most drifted features: write one sentence each explaining the business consequence for credit decisions -- not just 'distribution shifted'.


In [ ]:
# Exercise 2.2: PSI context
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: Population Stability Index --
import numpy as np, pandas as pd

def compute_psi(expected, actual, buckets=10):
    """PSI: < 0.1 stable | 0.1-0.25 moderate | > 0.25 severe"""
    bp = np.unique(np.percentile(expected, np.linspace(0, 100, buckets+1)))
    def props(d, b):
        c = np.histogram(d, bins=b)[0]
        return (c + 1e-6) / (len(d) + 1e-6 * len(c))
    p_exp, p_act = props(expected, bp), props(actual, bp)
    return float(np.sum((p_act - p_exp) * np.log(p_act / p_exp)))

q1 = pd.read_csv('datasets/creditdefault_q1.csv')
q4 = pd.read_csv('datasets/creditdefault_q4.csv')
features = [c for c in q1.columns if c != 'DEFAULT_NEXT_MONTH']
results = []
for f in features:
    psi = compute_psi(q1[f].dropna(), q4[f].dropna())
    severity = 'STABLE' if psi < 0.1 else 'MODERATE' if psi < 0.25 else 'SEVERE'
    results.append({'feature': f, 'psi': round(psi, 4), 'severity': severity})
df = pd.DataFrame(results).sort_values('psi', ascending=False)
print(df.to_string(index=False))


### Expected Output (illustrative)

```
      feature    psi severity
    LIMIT_BAL 0.3812   SEVERE
        PAY_0 0.1904 MODERATE
    BILL_AMT1 0.0743   STABLE
     PAY_AMT1 0.0518   STABLE
          AGE 0.0212   STABLE
          ...    ...      ...
```
`LIMIT_BAL` (0.3812) is well above the 0.25 severe threshold — the single feature to
investigate first in Week 3.


### Monday Morning Moment

*Slack -- Monday, 2:00pm.*

**Temi Adeyemi:** PSI for LIMIT_BAL: 0.38. Severe drift.

**Dr. Emeka Obi:** What changed about credit limits between Q1 and Q4?

**Temi Adeyemi:** Q4 has significantly more high-limit accounts -- above NT$400,000. In Q1 the distribution was concentrated below NT$200,000.

**Dr. Emeka Obi:** And the model's training data?

**Temi Adeyemi:** Q1 training data. The model has seen very few high-limit accounts. It cannot assess them accurately.

**Dr. Emeka Obi:** So it is approving high-limit accounts it was not trained to evaluate. Write that for Farida in plain language.

**Temi Adeyemi:** 'The model is being asked to assess credit profiles it has not seen before. Its predictions for high-limit accounts are less reliable than for standard-limit accounts.'

**Dr. Emeka Obi:** That is the sentence. Put a number on it.




---

## TUESDAY -- "Kolmogorov-Smirnov Test"


The KS test: H0 = both samples come from the same distribution. A p-value below the Bonferroni-corrected threshold rejects H0. PSI tells you how much drift occurred. KS tells you whether it is statistically significant given the sample sizes. With 30,000 records in CreditDefault-NG, even small shifts will be significant -- PSI matters more than the p-value for deciding whether to act.


### Exercise 2.3 -- KS with Bonferroni

Run KS tests for all features with Bonferroni correction. How many features remain significant after correction?


In [ ]:
# Exercise 2.3: KS with Bonferroni
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: Kolmogorov-Smirnov Test --
from scipy import stats
import pandas as pd

q1 = pd.read_csv('datasets/creditdefault_q1.csv')
q4 = pd.read_csv('datasets/creditdefault_q4.csv')
features = [c for c in q1.columns if c != 'DEFAULT_NEXT_MONTH']
alpha_corrected = 0.05 / len(features)  # Bonferroni correction

results = []
for feat in features:
    stat, p = stats.ks_2samp(q1[feat].dropna(), q4[feat].dropna())
    results.append({'feature': feat, 'ks_stat': round(stat,4),
                    'p_value': round(p,6), 'drifted': p < alpha_corrected})
ks_df = pd.DataFrame(results).sort_values('ks_stat', ascending=False)
print(f'Bonferroni alpha: {alpha_corrected:.5f}')
print(f'Drifted features: {ks_df.drifted.sum()}/{len(ks_df)}')
print(ks_df.to_string(index=False))


### Expected Output (illustrative)

```
Bonferroni alpha: 0.00227
Drifted features: 3/22
    feature  ks_stat  p_value  drifted
  LIMIT_BAL   0.2140  0.00000     True
      PAY_0   0.1180  0.00041     True
  BILL_AMT1   0.0690  0.03200    False
```
Only 3 features clear the Bonferroni-corrected threshold — the correction matters here:
without it, testing 22 features at uncorrected alpha=0.05 would flag roughly one feature as
"drifted" by chance alone, even with no real drift.



---

## WEDNESDAY -- "Evidently AI Drift Report"


Evidently generates a self-contained HTML drift report combining PSI, KS, Wasserstein distance, and distribution plots for every feature. Send the HTML to Farida. Use the structured JSON output for programmatic alerting.


### Exercise 2.4 -- Evidently report

Generate the DataDrift report. Open in browser. Find: highest Wasserstein distance, highest KS statistic, feature Evidently flags most critical. Are all three the same feature?


In [ ]:
# Exercise 2.4: Evidently report
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: Evidently AI Drift Report --
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
import pandas as pd

ref = pd.read_csv('datasets/creditdefault_q1.csv')
cur = pd.read_csv('datasets/creditdefault_q4.csv')
report = Report(metrics=[DataDriftPreset(), DataQualityPreset()])
report.run(reference_data=ref, current_data=cur)
report.save_html('outputs/drift_report_q1_vs_q4.html')
print('Saved: outputs/drift_report_q1_vs_q4.html')
result = report.as_dict()
summary = result['metrics'][0]['result']
print(f"Drifted: {summary['number_of_drifted_columns']}/{summary['number_of_columns']} features")


### Expected Output (illustrative)

```
Saved: outputs/drift_report_q1_vs_q4.html
Drifted: 4/23 features
```
Evidently's HTML report is the artifact to actually open — it shows per-feature distribution
plots, not just the drift count. Attach it to Friday's summary for Farida.



---

## THURSDAY -- "Concept Drift vs Data Drift"


Data drift: input feature distributions changed -- detectable from features alone. Concept drift: the relationship between features and the target changed -- requires labelled recent data to detect. For credit defaults the label arrives 30-90 days after the prediction, making concept drift harder to catch early. A proxy: monitor the model's approval rate over time. If it rises with no model change, concept drift is a candidate explanation.

Pause and Predict -- which of the two is harder to detect without labels? Which causes more damage if undetected for four months? Write your reasoning before checking the Common Mistakes section.


### STOP AND TRACE

Before running:

bp = np.unique(np.percentile(expected, np.linspace(0, 100, 11)))

What does np.linspace(0, 100, 11) return?
What does np.unique do here -- when would percentile values be non-unique?
What happens to PSI if two bins are empty because of a spike at a single value?
Why this matters: for payment status features like PAY_0 with only a few unique values (-2, -1, 0, 1, 2), many percentile-based bins will collapse to the same breakpoint. np.unique prevents empty bins. The epsilon (1e-6) prevents log(0) errors.


In [ ]:
# Exercise 2.5: STOP AND TRACE -- PSI bins
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Concept Drift vs Data Drift --
# Data drift: done above with PSI and KS tests

# Concept drift proxy: track model approval rate over time
import pandas as pd
import mlflow.sklearn

model = mlflow.sklearn.load_model('models:/credit-default-v2/Production')
for q, path in [('Q1', 'datasets/creditdefault_q1.csv'),
                ('Q4', 'datasets/creditdefault_q4.csv')]:
    df = pd.read_csv(path)
    X  = df.drop('DEFAULT_NEXT_MONTH', axis=1)
    prob = model.predict_proba(X)[:, 1]
    print(f'{q}: mean default prob={prob.mean():.4f}, '
          f'approval rate (< 0.35): {(prob < 0.35).mean():.1%}')


### Expected Output (illustrative)

```
Q1: mean default prob=0.1823, approval rate (< 0.35): 81.2%
Q4: mean default prob=0.1654, approval rate (< 0.35): 93.4%
```
The model is approving *more* in Q4 despite the world having become riskier for the
high-limit segment — that gap (81.2% -> 93.4%) is the concept-drift signature: the input
distribution shifted, and the model's decisions shifted with it in a direction that increases
business risk.



---

## FRIDAY -- "The Friday Build: Drift Summary for Farida"


Write the drift summary for Farida. No PSI values, no KS statistics. Three things: what changed in the data in plain English, which customer profiles are most affected, and what the recommended action is. Attach the Evidently HTML as an appendix. Test: read it aloud to someone who does not work in ML. If they can answer 'what happened and what are we doing about it', you have written it correctly.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Drift Summary for Farida --
# Drift summary structure for a non-technical PM:
# HEADLINE: one sentence -- what happened
# AFFECTED CUSTOMER PROFILES: which customer types, not which features
# BUSINESS IMPACT: effect on approval/decline decisions
# MODEL RELIABILITY: safe to use? for which customer types?
# ACTION: concrete next step with owner and timeline
# APPENDIX: outputs/drift_report_q1_vs_q4.html


### Expected Output (illustrative)

No code output — Friday's deliverable is the drift summary document itself. The headline
sentence that structure is built for: *"Approval rates rose because credit-limit profiles
have shifted since Q1 launch, not because risk actually decreased — reliability is degraded
for higher-limit customers specifically."*


### Friday Workplace Moment

*DataFlow Infrastructure -- Friday standup.*

**Farida Usman:** Which customer groups are most affected?

**Temi Adeyemi:** Two groups: customers with credit limits above NT$400,000, and customers with two-month payment delays in PAY_0. Both are more common in Q4 than in Q1. The model was not trained on enough of either.

**Farida Usman:** How many Q4 decisions fall into those groups?

**Temi Adeyemi:** Approximately 1,840 customers -- about 12% of Q4 decisions.

**Farida Usman:** Flag those for manual review.

**Dr. Emeka Obi:** Temi: flagging script by Monday. Sola: set up the review queue.



## YOUR WEEK 2 CHECKLIST

Check each item before moving on.

- [ ] I can compute PSI from scratch without referring to notes.
- [ ] I know the difference between data drift (inputs changed) and concept drift (relationship changed).
- [ ] The Evidently HTML report is saved and readable by a non-technical stakeholder.
- [ ] I can state the three most drifted features and their business implications in one sentence each.
- [ ] My drift summary for Farida contains no statistical jargon.


## Challenge Task

> **Core path:** Implement CUSUM drift detection on LIMIT_BAL using a rolling monthly window. At which month does CUSUM first signal an alarm?

> **Advanced path:** Compute Jensen-Shannon divergence for each feature. Compare the ranking to PSI. Where do they disagree and when would you prefer JS divergence?


## Common Mistakes

**Treating PSI as a model performance metric:** PSI measures input drift, not prediction quality. Always cross-reference drifted features with the model's SHAP importances before alarming.

**Ignoring multiple testing correction:** Running KS tests on 23 features without Bonferroni correction produces ~1.15 false positives on average. Report corrected p-values.

**Reporting PSI values to a PM:** PSI=0.38 means nothing to Farida. '12% of Q4 customers fall into profiles the model was not trained on' is actionable.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)